# Tahap 2: Case Representation
**Mata Kuliah:** Penalaran Komputer — SubCPMK-3 Case-Based Reasoning  
**Domain:** Perlindungan Anak  
**Tujuan:** Ekstraksi metadata & fitur teks dari corpus putusan, simpan ke `cases.csv` dan `cases.json`

### Alur Notebook:
1. Install & Import
2. Konfigurasi & Load Data dari Tahap 1
3. Koneksi Groq API
4. Ekstraksi Metadata per Dokumen (via Groq + fallback regex)
5. Feature Engineering
6. Simpan ke CSV & JSON
7. Validasi & Preview Hasil

## Cell 1 — Install Dependencies

In [ ]:
!pip install -q groq pandas tqdm

## Cell 2 — Import Library

In [ ]:
import os
import re
import json
import time
import logging
from pathlib import Path
from getpass import getpass

import pandas as pd
from tqdm import tqdm
from groq import Groq

print("✅ Library berhasil diimport")

## Cell 3 — Konfigurasi

In [ ]:
# ============================================================
# KONFIGURASI — sesuaikan dengan hasil Tahap 1
# ============================================================
CONFIG = {
    "DOMAIN"          : "Perlindungan Anak",
    "GROQ_MODEL"      : "llama-3.3-70b-versatile",  # model gratis Groq
    "GROQ_DELAY"      : 1.5,   # detik antar request (hindari rate limit)
    "MAX_CHARS_INPUT" : 6000,  # batas karakter teks yang dikirim ke Groq
    "TOP_K_PREVIEW"   : 5,     # jumlah preview di akhir
}

# Direktori (harus sama dengan Tahap 1)
BASE_DIR   = Path(".").resolve()
DATA_RAW   = BASE_DIR / "data" / "raw"
DATA_PROC  = BASE_DIR / "data" / "processed"
LOGS_DIR   = BASE_DIR / "logs"
META_FILE  = BASE_DIR / "data" / "metadata_with_pdf.json"

DATA_PROC.mkdir(parents=True, exist_ok=True)

# Setup logging
logging.basicConfig(
    level   = logging.INFO,
    format  = "%(asctime)s | %(levelname)s | %(message)s",
    handlers=[
        logging.FileHandler(LOGS_DIR / "representation.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print(f"📁 Raw text : {DATA_RAW}")
print(f"📁 Processed: {DATA_PROC}")

## Cell 4 — Load Data dari Tahap 1

In [ ]:
# Load metadata dari Tahap 1
assert META_FILE.exists(), f"❌ {META_FILE} tidak ditemukan. Jalankan Tahap 1 dulu."

with open(META_FILE, encoding="utf-8") as f:
    metadata_raw = json.load(f)

# Filter hanya yang berhasil dan punya file teks
dokumen_valid = []
for item in metadata_raw:
    case_id  = item.get("case_id", "")
    path_txt = DATA_RAW / f"{case_id}.txt"
    if item.get("status_pdf") in ("berhasil", "sudah_ada") and path_txt.exists():
        item["path_txt"] = str(path_txt)
        dokumen_valid.append(item)

print(f"✅ Total dokumen valid dari Tahap 1 : {len(dokumen_valid)}")
print(f"⚠️  Dokumen tanpa teks (skip)        : {len(metadata_raw) - len(dokumen_valid)}")

if len(dokumen_valid) < 30:
    print(f"\n⚠️  Perhatian: dokumen valid ({len(dokumen_valid)}) masih di bawah 30.")
    print(f"   Pastikan Tahap 1 sudah selesai sebelum lanjut.")
else:
    print(f"\n✅ Siap diproses!")

## Cell 5 — Koneksi Groq API

> Daftar gratis di https://console.groq.com → pilih **API Keys** → **Create API Key**  
> Paste key-nya di bawah saat diminta. Key tidak akan ditampilkan di layar.

In [ ]:
# Masukkan API key Groq
# Cara aman: pakai getpass() agar key tidak terlihat di output
GROQ_API_KEY = getpass("Masukkan Groq API Key: ")

client = Groq(api_key=GROQ_API_KEY)

# Test koneksi
try:
    test = client.chat.completions.create(
        model    = CONFIG["GROQ_MODEL"],
        messages = [{"role": "user", "content": "Balas hanya dengan kata: OK"}],
        max_tokens = 5,
    )
    print(f"✅ Groq terhubung — model: {CONFIG['GROQ_MODEL']}")
    print(f"   Response test: {test.choices[0].message.content.strip()}")
except Exception as e:
    print(f"❌ Koneksi Groq gagal: {e}")
    print(f"   Pastikan API key benar dan ada koneksi internet.")

## Cell 6 — Prompt & Fungsi Ekstraksi Metadata

In [ ]:
# ============================================================
# PROMPT EKSTRAKSI METADATA
# Groq diminta mengembalikan JSON murni — tidak ada teks lain
# ============================================================

SYSTEM_PROMPT = """Kamu adalah asisten ekstraksi data hukum Indonesia.
Tugasmu: ekstrak metadata dari teks putusan pengadilan domain Perlindungan Anak.
Kembalikan HANYA JSON valid tanpa penjelasan, tanpa markdown, tanpa komentar.
Jika suatu field tidak ditemukan, isi dengan string kosong "".
""".strip()

USER_PROMPT_TEMPLATE = """Ekstrak metadata berikut dari teks putusan ini:

{
  "no_perkara"        : "nomor perkara (contoh: 10/Pid.Sus/2023/PN Mlg)",
  "tanggal_putusan"   : "tanggal putusan (format: YYYY-MM-DD atau DD Bulan YYYY)",
  "pengadilan"        : "nama pengadilan yang memutus",
  "terdakwa"          : "nama terdakwa / para terdakwa",
  "jaksa"             : "nama jaksa penuntut umum",
  "pasal_dakwaan"     : "pasal-pasal yang didakwakan (contoh: Pasal 76C jo Pasal 80 UU No.35 Tahun 2014)",
  "pasal_terbukti"    : "pasal yang terbukti dalam putusan",
  "amar_putusan"      : "isi amar putusan singkat (terbukti/tidak, vonis jika ada)",
  "vonis_penjara"     : "lama hukuman penjara jika ada (contoh: 5 tahun), kosong jika tidak ada",
  "vonis_denda"       : "jumlah denda jika ada, kosong jika tidak ada",
  "barang_bukti"      : "ringkasan barang bukti utama (maksimal 1 kalimat)",
  "ringkasan_fakta"   : "ringkasan fakta hukum utama (maksimal 2 kalimat)"
}

TEKS PUTUSAN:
"""
{teks}
"""

Kembalikan hanya JSON di atas, diisi dengan data dari teks."""


def ekstrak_via_groq(teks: str, case_id: str) -> dict:
    """
    Kirim teks ke Groq, dapatkan metadata dalam format JSON.
    Returns dict metadata atau dict kosong jika gagal.
    """
    # Potong teks jika terlalu panjang
    teks_input = teks[:CONFIG["MAX_CHARS_INPUT"]]

    prompt = USER_PROMPT_TEMPLATE.format(teks=teks_input)

    try:
        response = client.chat.completions.create(
            model    = CONFIG["GROQ_MODEL"],
            messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user",   "content": prompt},
            ],
            max_tokens  = 800,
            temperature = 0.0,  # deterministik untuk ekstraksi
        )
        raw = response.choices[0].message.content.strip()

        # Bersihkan jika ada markdown fence
        raw = re.sub(r"^```(?:json)?\s*", "", raw, flags=re.MULTILINE)
        raw = re.sub(r"\s*```$",          "", raw, flags=re.MULTILINE)
        raw = raw.strip()

        hasil = json.loads(raw)
        logger.info(f"{case_id}: ✅ Groq OK — no_perkara={hasil.get('no_perkara','?')}")
        return hasil

    except json.JSONDecodeError as e:
        logger.warning(f"{case_id}: JSON decode error — {e} | raw={raw[:100]}")
        return {}
    except Exception as e:
        logger.error(f"{case_id}: Groq error — {e}")
        return {}


print("✅ Prompt dan fungsi ekstraksi siap")

## Cell 7 — Fallback Regex

> Dipakai bila Groq gagal (rate limit, koneksi, dll)  
> Pola regex disesuaikan dengan struktur putusan MA RI domain Perlindungan Anak

In [ ]:
BULAN_ID = (
    "Januari|Februari|Maret|April|Mei|Juni|"
    "Juli|Agustus|September|Oktober|November|Desember"
)

def first_match(patterns: list, text: str, group: int = 1) -> str:
    """Coba semua pola, kembalikan match pertama yang ditemukan."""
    for pat in patterns:
        m = re.search(pat, text, re.IGNORECASE | re.MULTILINE | re.DOTALL)
        if m:
            val = m.group(group).strip()
            return re.sub(r"\s+", " ", val)
    return ""


def ekstrak_via_regex(teks: str) -> dict:
    """
    Ekstrak metadata dengan regex sebagai fallback.
    Lebih rapuh dari Groq tapi tidak butuh API.
    """
    return {
        "no_perkara": first_match([
            r"(?:Nomor|No\.?)\s*[:\.\-]?\s*(\d+[/\-][A-Za-z.]+[/\-]\d{4}[/\-][A-Z.\s]+)",
            r"(\d+\s*/\s*Pid\.Sus[^\n]{0,40})",
        ], teks),

        "tanggal_putusan": first_match([
            r"(?:tanggal|pada|hari)\s+(?:\w+,\s*)?(\d{1,2}\s+(?:" + BULAN_ID + r")\s+\d{4})",
            r"(\d{4}-\d{2}-\d{2})",
        ], teks),

        "pengadilan": first_match([
            r"PENGADILAN\s+(NEGERI|TINGGI)\s+([A-Z\s]+)",
            r"(?:di|pada)\s+(Pengadilan\s+(?:Negeri|Tinggi)\s+[A-Z][a-zA-Z\s]+)",
        ], teks),

        "terdakwa": first_match([
            r"(?:Terdakwa|TERDAKWA)\s*[:\-]?\s*([A-Z][A-Za-z\s,./]+?)(?=\n|;|,\s*(?:bin|binti|alias))",
            r"(?:Nama\s*Lengkap|Nama)\s*[:\-]\s*([A-Z][A-Za-z\s]+)",
        ], teks),

        "jaksa": first_match([
            r"(?:Penuntut Umum|Jaksa)\s*[:\-]?\s*([A-Z][A-Za-z\s,.]+?)(?=\n|;)",
        ], teks),

        "pasal_dakwaan": first_match([
            r"(?:Pasal|pasal)\s+(\d+[A-Za-z]?(?:\s*(?:jo\.?|dan|atau|junto)\s*(?:Pasal|pasal)\s*\d+[A-Za-z]?)*[^\n]{0,100}(?:Undang|UU|KUHP)[^\n]{0,80})",
            r"(?:melanggar|terbukti)\s+(Pasal[^\n]{0,120})",
        ], teks),

        "pasal_terbukti": first_match([
            r"(?:terbukti|dinyatakan terbukti)[^\n]{0,30}(Pasal[^\n]{0,120})",
        ], teks),

        "amar_putusan": first_match([
            r"MENGADILI\s*[:\-]?\s*(.{20,300}?)(?=\n\n|Demikian|Ditetapkan)",
            r"(?:menyatakan|menjatuhkan hukuman)[^\n]{0,200}",
        ], teks),

        "vonis_penjara": first_match([
            r"(\d+\s*(?:tahun|bulan)\s*(?:dan\s*\d+\s*bulan)?)[^\n]{0,30}(?:penjara|pidana)",
            r"pidana\s+penjara\s+selama\s+(\d+[^\n]{0,30})",
        ], teks),

        "vonis_denda": first_match([
            r"denda\s+(?:sebesar\s+)?(?:Rp\.?\s*)?((?:Rp\.?\s*)?[\d.,]+[^\n]{0,30})",
        ], teks),

        "barang_bukti"   : "",
        "ringkasan_fakta": "",
    }


print("✅ Fungsi fallback regex siap")

## Cell 8 — Feature Engineering

In [ ]:
def hitung_fitur(teks: str) -> dict:
    """
    Hitung fitur statistik sederhana dari teks putusan.
    Dipakai sebagai tambahan representasi di samping metadata.
    """
    kata       = teks.split()
    kalimat    = re.split(r"[.!?]+", teks)
    paragraf   = [p for p in teks.split("\n\n") if p.strip()]

    # Deteksi apakah ada kata kunci perlindungan anak
    kata_kunci_pa = [
        "anak", "korban", "kekerasan", "seksual", "eksploitasi",
        "trafficking", "perdagangan", "perlindungan", "minor",
        "di bawah umur", "pencabulan", "pemerkosaan", "penganiayaan"
    ]
    teks_lower = teks.lower()
    kata_kunci_ditemukan = [k for k in kata_kunci_pa if k in teks_lower]

    return {
        "jumlah_kata"      : len(kata),
        "jumlah_kalimat"   : len([k for k in kalimat if k.strip()]),
        "jumlah_paragraf"  : len(paragraf),
        "avg_kata_kalimat" : round(len(kata) / max(len(kalimat), 1), 1),
        "kata_kunci_pa"    : ", ".join(kata_kunci_ditemukan),
        "n_kata_kunci_pa"  : len(kata_kunci_ditemukan),
    }

print("✅ Fungsi feature engineering siap")

## Cell 9 — Pipeline Utama: Ekstraksi + Feature Engineering

In [ ]:
def pipeline_tahap2(dokumen_list: list) -> list:
    """
    Pipeline lengkap Tahap 2:
    Teks bersih → Groq/regex → metadata + fitur → list of dict

    Setiap record berisi:
    - identitas (case_id, no_perkara, tanggal, dst)
    - metadata hukum (pasal, terdakwa, amar, vonis, dst)
    - fitur teks (jumlah kata, kata kunci, dst)
    - teks_full (teks lengkap, untuk embedding di Tahap 3)
    """
    records  = []
    n_groq   = 0
    n_regex  = 0
    n_gagal  = 0

    print(f"🔄 Memproses {len(dokumen_list)} dokumen...\n")

    for item in tqdm(dokumen_list, desc="Ekstraksi"):
        case_id  = item["case_id"]
        path_txt = Path(item["path_txt"])

        # Baca teks
        try:
            teks = path_txt.read_text(encoding="utf-8", errors="ignore").strip()
        except Exception as e:
            logger.error(f"{case_id}: gagal baca teks — {e}")
            n_gagal += 1
            continue

        if not teks or len(teks.split()) < 50:
            logger.warning(f"{case_id}: teks terlalu pendek, skip")
            n_gagal += 1
            continue

        # ── Ekstraksi metadata ──────────────────────────────────────
        metadata = ekstrak_via_groq(teks, case_id)

        if not metadata or not any(metadata.values()):
            # Fallback ke regex
            logger.warning(f"{case_id}: Groq gagal/kosong → pakai regex")
            metadata = ekstrak_via_regex(teks)
            n_regex += 1
        else:
            n_groq += 1

        # Delay agar tidak kena rate limit Groq
        time.sleep(CONFIG["GROQ_DELAY"])

        # ── Feature engineering ─────────────────────────────────────
        fitur = hitung_fitur(teks)

        # ── Susun record ────────────────────────────────────────────
        record = {
            # Identitas
            "case_id"          : case_id,
            "url_detail"       : item.get("url_detail", ""),
            "path_txt"         : str(path_txt),

            # Metadata dari Groq/regex
            "no_perkara"       : metadata.get("no_perkara", ""),
            "tanggal_putusan"  : metadata.get("tanggal_putusan", ""),
            "pengadilan"       : metadata.get("pengadilan", ""),
            "terdakwa"         : metadata.get("terdakwa", ""),
            "jaksa"            : metadata.get("jaksa", ""),
            "pasal_dakwaan"    : metadata.get("pasal_dakwaan", ""),
            "pasal_terbukti"   : metadata.get("pasal_terbukti", ""),
            "amar_putusan"     : metadata.get("amar_putusan", ""),
            "vonis_penjara"    : metadata.get("vonis_penjara", ""),
            "vonis_denda"      : metadata.get("vonis_denda", ""),
            "barang_bukti"     : metadata.get("barang_bukti", ""),
            "ringkasan_fakta"  : metadata.get("ringkasan_fakta", ""),

            # Fitur teks
            "jumlah_kata"      : fitur["jumlah_kata"],
            "jumlah_kalimat"   : fitur["jumlah_kalimat"],
            "jumlah_paragraf"  : fitur["jumlah_paragraf"],
            "avg_kata_kalimat" : fitur["avg_kata_kalimat"],
            "kata_kunci_pa"    : fitur["kata_kunci_pa"],
            "n_kata_kunci_pa"  : fitur["n_kata_kunci_pa"],

            # Sumber ekstraksi
            "sumber_ekstraksi" : "groq" if n_groq > n_regex else "regex",

            # Teks lengkap (dibutuhkan Tahap 3 untuk embedding)
            "teks_full"        : teks,
        }
        records.append(record)

    print(f"\n{'='*50}")
    print(f"  REKAP PIPELINE TAHAP 2")
    print(f"{'='*50}")
    print(f"  ✅ Total record    : {len(records)}")
    print(f"  🤖 Via Groq        : {n_groq}")
    print(f"  🔤 Via regex       : {n_regex}")
    print(f"  ❌ Gagal           : {n_gagal}")
    print(f"{'='*50}")

    return records

print("✅ Fungsi pipeline_tahap2 siap")

In [ ]:
# Jalankan pipeline
records = pipeline_tahap2(dokumen_valid)

## Cell 10 — Simpan ke CSV & JSON

In [ ]:
assert records, "❌ Tidak ada record — jalankan Cell 9 dulu"

# ── Simpan CSV ──────────────────────────────────────────────
# CSV tanpa kolom teks_full (terlalu besar, ada di JSON)
df = pd.DataFrame(records)
kolom_csv = [c for c in df.columns if c != "teks_full"]
df_csv    = df[kolom_csv]

csv_path = DATA_PROC / "cases.csv"
df_csv.to_csv(csv_path, index=False, encoding="utf-8-sig")
print(f"✅ CSV disimpan: {csv_path}  ({len(df_csv)} baris, {len(kolom_csv)} kolom)")

# ── Simpan JSON (lengkap dengan teks_full) ──────────────────
json_path = DATA_PROC / "cases.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(records, f, ensure_ascii=False, indent=2)
print(f"✅ JSON disimpan: {json_path}  ({len(records)} record)")

# Ukuran file
print(f"\n  Ukuran CSV  : {csv_path.stat().st_size / 1024:.1f} KB")
print(f"  Ukuran JSON : {json_path.stat().st_size / 1024:.1f} KB")

## Cell 11 — Validasi Kelengkapan Metadata

In [ ]:
df = pd.read_csv(DATA_PROC / "cases.csv")

print(f"{'='*55}")
print(f"  LAPORAN VALIDASI — TAHAP 2")
print(f"{'='*55}")
print(f"  Total kasus   : {len(df)}")
print(f"{'='*55}")

# Cek kelengkapan per kolom metadata penting
kolom_penting = [
    "no_perkara", "tanggal_putusan", "pengadilan",
    "terdakwa", "pasal_dakwaan", "amar_putusan",
    "vonis_penjara", "ringkasan_fakta"
]

print(f"\n  Kelengkapan metadata:")
for k in kolom_penting:
    if k in df.columns:
        terisi = df[k].astype(str).str.strip().ne("").sum()
        pct    = terisi / len(df) * 100
        bar    = "█" * int(pct // 10) + "░" * (10 - int(pct // 10))
        print(f"    {k:<20} [{bar}] {pct:5.1f}%  ({terisi}/{len(df)})")

print(f"\n  Sumber ekstraksi:")
if "sumber_ekstraksi" in df.columns:
    print(df["sumber_ekstraksi"].value_counts().to_string())

print(f"\n  Statistik jumlah kata:")
print(f"    Min  : {df['jumlah_kata'].min():,}")
print(f"    Max  : {df['jumlah_kata'].max():,}")
print(f"    Rata : {df['jumlah_kata'].mean():.0f}")
print(f"{'='*55}")

## Cell 12 — Preview Hasil

In [ ]:
df = pd.read_csv(DATA_PROC / "cases.csv")

print(f"📋 Preview {CONFIG['TOP_K_PREVIEW']} kasus pertama:\n")
for _, row in df.head(CONFIG["TOP_K_PREVIEW"]).iterrows():
    print(f"  [{row['case_id']}]")
    print(f"    No. Perkara   : {row.get('no_perkara', '-')}")
    print(f"    Tanggal       : {row.get('tanggal_putusan', '-')}")
    print(f"    Pengadilan    : {row.get('pengadilan', '-')}")
    print(f"    Terdakwa      : {str(row.get('terdakwa', '-'))[:60]}")
    print(f"    Pasal Dakwaan : {str(row.get('pasal_dakwaan', '-'))[:80]}")
    print(f"    Amar Putusan  : {str(row.get('amar_putusan', '-'))[:80]}")
    print(f"    Vonis         : {row.get('vonis_penjara', '-')}")
    print(f"    Kata kunci PA : {row.get('kata_kunci_pa', '-')}")
    print()

print(f"\n✅ Tahap 2 selesai — lanjut ke 03_retrieval.ipynb")

---
## ✅ Tahap 2 Selesai!

**Output yang dihasilkan:**
```
/data/processed/
├── cases.csv    ← metadata + fitur (tanpa teks_full)
└── cases.json   ← lengkap dengan teks_full (untuk Tahap 3)
/logs/
└── representation.log
```

**Kolom di `cases.csv`:**
| Kolom | Keterangan |
|-------|-----------|
| case_id | ID unik dokumen |
| no_perkara | Nomor perkara pengadilan |
| tanggal_putusan | Tanggal putusan dijatuhkan |
| pengadilan | Nama pengadilan |
| terdakwa | Nama terdakwa |
| pasal_dakwaan | Pasal yang didakwakan |
| pasal_terbukti | Pasal yang terbukti di putusan |
| amar_putusan | Isi amar putusan singkat |
| vonis_penjara | Lama hukuman penjara |
| vonis_denda | Jumlah denda |
| barang_bukti | Ringkasan barang bukti |
| ringkasan_fakta | Ringkasan fakta hukum |
| jumlah_kata | Jumlah kata dokumen |
| n_kata_kunci_pa | Jumlah kata kunci perlindungan anak |

**Langkah selanjutnya:** Buka `03_retrieval.ipynb` untuk Case Retrieval.